HW 1 Miriam Zuckerbrot

Task 1: Evaluation Rubric
1. Criterion Definitions

- Fluency

Good: Sentences flow naturally; uses varied structures; sounds human.

OK: Understandable but slightly repetitive or robotic phrasing.

Bad: Disjointed sentences; difficult to read or nonsensical.

- Grammar

Good: Zero errors in spelling, punctuation, or syntax.

OK: 1–2 minor typos or punctuation slips.

Bad: 3+ errors or major syntax issues impacting professionalism.

- Tone

Good: Perfectly balances warmth/friendliness with credibility.

OK: Slightly too formal or too "salesy," but generally appropriate.

Bad: Inappropriate tone (e.g., aggressive, cold, or clinical).

- Length

Good: 50–90 words (inclusive).

OK: 40–49 or 91–110 words.

Bad: < 40 or > 110 words.

- Grounding

Good: 100% of facts are derived from provided features.

OK: Includes 1 minor "fluff" claim (e.g., "best in class") not in data.

Bad: Includes hallucinated facts (wrong material, warranty, or attributes).

- Latency

Good: Full response in < 1500 ms.

OK: Full response in 1500–3000 ms.

Bad: Full response takes > 3000 ms.

- Cost

Good: < $0.002 per 1K tokens.

OK: $0.002–$0.01 per 1K tokens.

Bad: > $0.01 per 1K tokens.

2. Pass / Fail Definition

A. Cumulative Pass Bar
To pass, a description must achieve:

At least 5 "Good" ratings across the 7 criteria.

B. Go / No-Go Rules
The description is an automatic FAIL if any of the following occur:

Grounding is not Good.

Grammar is Bad.

Tone is Bad.

Task 2

In [ ]:
import os
import pandas as pd
import time
import requests
from openai import OpenAI
import openpyxl

In [9]:
df_path = "Assignment_01_product_dataset.csv" 

client = OpenAI(
    base_url="https://api.tokenfactory.nebius.com/v1/",
    api_key=os.environ.get("NEBIUS_API_KEY")
)

Prompting guidlines:

Be Concise: Use as few words as possible, avoid stating the obvious, and refrain from lengthy instructions that the model might ignore.

Use Positive Phrasing: Focus on the "DOs" rather than the "DON'Ts." 

Allow "I don't know": Explicitly give the model permission to reply with "I don't know" if the answer is not contained in the provided context.

Optimize the System Prompt: Keep the system prompt short and general. Use it to define the model's role/persona, behavioral rules, output format, and guardrails. Do not put the actual task or data that changes per request in the system prompt; those belong in the user message.

In [ ]:
def generate_product_data(row, client):
    system_msg = (
        "You are a professional e-commerce copywriter." # role
        "Write friendly, engaging, and persuasive product descriptions based strictly on provided data." # behavior
        "Keep sentences fluent, natural, and grammatically correct." # behavior
        "If information is missing, state 'I do not know'." # gaurdrail
        "Output only the description." # output
    ) 
    
    # User message contains the task and the specific data
    user_msg = f"""Write a 50–90 word product description using these details:

    Name: {row['product_name']}
    Attributes: {row['Product_attribute_list']}
    Material: {row['material']}
    Warranty: {row['warranty']}

    Structure the response with an engaging opening, a benefit-focused body, and a closing mention of the warranty."""

    start_time = time.time()
    
    response = client.chat.completions.create(
        model="meta-llama/Meta-Llama-3.1-8B-Instruct",
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg}
        ],
        temperature=0.7 # Slight creativity for 'persuasive' tone
    )
    
    end_time = time.time()
    description = response.choices[0].message.content
    word_count = len(description.split())
    
    return {
        "generated_description": description,
        "word_count": word_count, # Added for easier evaluation
        "latency_ms": int((end_time - start_time) * 1000),
        "input_tokens": response.usage.prompt_tokens,
        "output_tokens": response.usage.completion_tokens
    }


def calculate_cost(input_tokens, output_tokens):
    # Rates per 1,000,000 tokens
    input_rate = 0.02 / 1_000_000
    output_rate = 0.06 / 1_000_000
    
    total_cost = (input_tokens * input_rate) + (output_tokens * output_rate)
    return round(total_cost, 6) # Round to 6 decimal places for clarity



df = pd.read_csv(df_path)
results = []
for index, row in df.iterrows():
    res = generate_product_data(row, client)
    
    call_cost = calculate_cost(res['input_tokens'], res['output_tokens'])

    row_data = {
        **row.to_dict(),
        **res,
        "cost_usd": call_cost,
        
    }
    results.append(row_data)

# Create Final DataFrame
final_df = pd.DataFrame(results)

# Add Blank Rubric Columns for Task 3/4
rubric_cols = ["Fluency", "Grammar", "Tone", "Length", "Grounding", "Latency", "Cost", "final_score"]
for col in rubric_cols:
    final_df[col] = ""

# Save to Excel
final_df.to_excel("assignment_01.xlsx", index=False)

Task 3 - xlsx

Task 4

Looking at results from task 3, it seems that the main issue is Grounding. The modle is assuming / hallucinating traits that are not listed in the input.

Experiment 1

Change: Modify the system prompt to include negative constraints and give an example.

Reason: Instead of just saying "be persuasive," we will explicitly forbid common filler phrases and require the model to "anchor" every sentence to a specific attribute.

In [16]:
def generate_product_data(row, client):
    system_msg = (
        "You are a professional e-commerce copywriter." # role
        "Write friendly, engaging, and persuasive product descriptions based strictly on provided data." # behavior
        "Keep sentences fluent, natural, and grammatically correct." # behavior
        "CRITICAL RULE: Do not use claims not found in the attributes "
        "(e.g., do not say 'premium' or 'best-in-class' unless stated)"
        "If information is missing, state 'I do not know'." # gaurdrail
        "Output only the description." # output
    ) 
    
    # User message contains the task and the specific data
    user_msg = f"""Write a 50–90 word product description using these details:

    Name: {row['product_name']}
    Attributes: {row['Product_attribute_list']}
    Material: {row['material']}
    Warranty: {row['warranty']}

    Structure the response with an engaging opening, a benefit-focused body, and a closing mention of the warranty."""

    start_time = time.time()
    
    response = client.chat.completions.create(
        model="meta-llama/Meta-Llama-3.1-8B-Instruct",
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg}
        ],
        temperature=0.7 # Slight creativity for 'persuasive' tone
    )
    
    end_time = time.time()
    description = response.choices[0].message.content
    word_count = len(description.split())
    
    return {
        "generated_description": description,
        "word_count": word_count, # Added for easier evaluation
        "latency_ms": int((end_time - start_time) * 1000),
        "input_tokens": response.usage.prompt_tokens,
        "output_tokens": response.usage.completion_tokens
    }


def calculate_cost(input_tokens, output_tokens):
    # Rates per 1,000,000 tokens
    input_rate = 0.02 / 1_000_000
    output_rate = 0.06 / 1_000_000
    
    total_cost = (input_tokens * input_rate) + (output_tokens * output_rate)
    return round(total_cost, 6) # Round to 6 decimal places for clarity



df = pd.read_csv(df_path)
results = []
for index, row in df.iterrows():
    res = generate_product_data(row, client)
    
    call_cost = calculate_cost(res['input_tokens'], res['output_tokens'])

    row_data = {
        **row.to_dict(),
        **res,
        "cost_usd": call_cost,
        
    }
    results.append(row_data)

# Create Final DataFrame
final_df = pd.DataFrame(results)

# Add Blank Rubric Columns for Task 3/4
rubric_cols = ["Fluency", "Grammar", "Tone", "Length", "Grounding", "Latency", "Cost", "final_score"]
for col in rubric_cols:
    final_df[col] = ""

# Save to Excel
final_df.to_excel("assignment_01_exp1.xlsx", index=False)

Results

The model in fact generated grounded results, though the price is much higher latency! and the descriptions are a bit more boring.

Experiment 2

Change: Lower the temperature from 0.7 to 0.4 and tighten top_p to 0.9, limit max_tokens to 150.

Reason: 

Temperature: At 0.7, the model has a higher probability of choosing "creative" tokens that may not be based on the input data. Lowering this makes the model more deterministic.

Top_p (Nucleus Sampling): By setting this to 0.9, the model will only consider the 90% most likely token candidates, cutting off the creative but potentially hallucinated words. Same for limiting tokens.

Result: The descriptions were more grounded than the original, but still not enough. For grounded we are only willing to take "Good" results. So many of the descriptions still didn't pass.

Experiment 3

Change: Lower the temperature from 0.7 to 0.1 and tighten top_p to 0.8, limit max_tokens to 140.

Reason: same as above, but more strict

In [21]:
def generate_product_data(row, client):
    system_msg = (
        "You are a professional e-commerce copywriter." # role
        "Write friendly, engaging, and persuasive product descriptions based strictly on provided data." # behavior
        "Keep sentences fluent, natural, and grammatically correct." # behavior
        "If information is missing, state 'I do not know'." # gaurdrail
        "Output only the description." # output
    ) 
    
    # User message contains the task and the specific data
    user_msg = f"""Write a 50–90 word product description using these details:

    Name: {row['product_name']}
    Attributes: {row['Product_attribute_list']}
    Material: {row['material']}
    Warranty: {row['warranty']}

    Structure the response with an engaging opening, a benefit-focused body, and a closing mention of the warranty."""

    start_time = time.time()
    
    response = client.chat.completions.create(
        model="meta-llama/Meta-Llama-3.1-8B-Instruct",
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg}
        ],
        temperature=0.1, # Slightly less creative for more grounded descriptions
        top_p=0.8,       # Nucleus sampling to cut off unlikely/random tokens
        max_tokens=140   # Safety cap to ensure it doesn't ramble
    )
    
    end_time = time.time()
    description = response.choices[0].message.content
    word_count = len(description.split())
    
    return {
        "generated_description": description,
        "word_count": word_count, # Added for easier evaluation
        "latency_ms": int((end_time - start_time) * 1000),
        "input_tokens": response.usage.prompt_tokens,
        "output_tokens": response.usage.completion_tokens
    }


def calculate_cost(input_tokens, output_tokens):
    # Rates per 1,000,000 tokens
    input_rate = 0.02 / 1_000_000
    output_rate = 0.06 / 1_000_000
    
    total_cost = (input_tokens * input_rate) + (output_tokens * output_rate)
    return round(total_cost, 6) # Round to 6 decimal places for clarity



df = pd.read_csv(df_path)
results = []
for index, row in df.iterrows():
    res = generate_product_data(row, client)
    
    call_cost = calculate_cost(res['input_tokens'], res['output_tokens'])

    row_data = {
        **row.to_dict(),
        **res,
        "cost_usd": call_cost,
        
    }
    results.append(row_data)

# Create Final DataFrame
final_df = pd.DataFrame(results)

# Add Blank Rubric Columns for Task 3/4
rubric_cols = ["Fluency", "Grammar", "Tone", "Length", "Grounding", "Latency", "Cost", "final_score"]
for col in rubric_cols:
    final_df[col] = ""

# Save to Excel
final_df.to_excel("assignment_01_exp4.xlsx", index=False)

Results:
The descriptions were significantly more grounded than the original. Most of the descriptions passed. In addition, multiple latencies were better than those of Exp 1. 

Task 5

Forcing the model to write its reasoning first, before it commits to a verdict label, acts as chain-of-thought prompting.  The model cannot pick a verdict and then justify it, rather it must think through the evidence and only then decide on the verdict.

In [22]:
from pydantic import BaseModel, Field
from typing import List
from enum import Enum
import json

In [54]:
class Verdict(str, Enum):
    good = "good"
    ok = "ok"
    bad = "bad"

class CriterionResult(BaseModel):
    explanation: str = Field(..., description="1 sentence max. State the key evidence and your reasoning concisely.")
    verdict: Verdict

class JudgeResponse(BaseModel):
    fluency: CriterionResult
    grammar: CriterionResult
    tone: CriterionResult
    length: CriterionResult
    grounding: CriterionResult

def get_judge_prompt(product_row, generated_description):
    return f"""
    You are an expert E-commerce Quality Assurance Judge. 
    Evaluate the following product description based strictly on the provided rubric.

    ### SOURCE DATA:
    - Name: {product_row['product_name']}
    - Attributes: {product_row['Product_attribute_list']}
    - Material: {product_row['material']}
    - Warranty: {product_row['warranty']}

    ### GENERATED DESCRIPTION TO EVALUATE:
    "{generated_description}"

    ### RUBRIC DEFINITIONS:
    1. Fluency: Good (Natural/human), OK (Robotic/repetitive), Bad (Disjointed).
    2. Grammar: Good (0 errors), OK (1-2 minor), Bad (3+ or major).
    3. Tone: Good (Warm/credible), OK (Too formal/salesy), Bad (Inappropriate).
    4. Length: Good (50-90 words), OK (40-49 or 91-110), Bad (outside range).
    5. Grounding: every specific, testable claim must be traceable to the source data. 

    To evaluate, apply this test to each descriptive phrase:
    1. Is the claim specific and testable (a property, rating, or capability that could be verified or contradicted)?
     - If no (e.g. "stunning", "effortless", "ultimate"): skip, not a grounding issue.
     - If yes: proceed to step 2.
    2. Is it explicitly supported by the source data?
     - yes: grounded, fine.
     - no, but it is a well-known intrinsic property of a named material (e.g. "durable titanium"): ok, minor embellishment.
     - no, and it adds a product-specific claim not in the source (e.g. "shatter-resistant Ceramic Shield"): hallucination.

    good: all testable claims are grounded.
    ok:   one intrinsic-material embellishment; no product-specific additions.
    bad:  any product-specific claim not present in the source data.

    Example (bad): source says "Ceramic Shield glass", description says "shatter-resistant Ceramic Shield glass". 
    "Shatter-resistant" is a specific product claim absent from the source. verdict: bad.

    Example (ok): source says "titanium frame", description says "durable titanium frame". Durability is intrinsic to titanium as a material. verdict: ok.

    For each criterion, provide your reasoning FIRST, then the verdict.
    """

def run_judge_task(row, description, client):
    system_msg = (
        "You are a rigorous e-commerce copy evaluator." # role
        "Evaluate product descriptions against a provided set of facts, based strictly on the provided rubric." # behavior
        "Provide detailed reasoning for each verdict." # behavior
        "Use only the product data and description supplied by the user." # gaurdrail
        "If a claim is not in the data, it is a hallucination." # gaurdrail
        "You must output a structured JSON response. For each criterion:"
        "1. Provide a clear 'explanation' first. 1 sentence only. Maximum 20 words. State the key evidence and your reasoning concisely."
        "2. Provide the 'verdict' (good, ok, or bad)." # output
    )

    prompt = get_judge_prompt(row, description)
    
    response = client.beta.chat.completions.parse( # structured response parsing
        model="Qwen/Qwen3-30B-A3B-Instruct-2507", #"google/gemma-2-9b-it-fast", #
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": prompt}
        ],
        response_format=JudgeResponse,
        max_tokens      = 600,
    )
    
    return response.choices[0].message.parsed

Task 6

After running a sanity check, I adjusted the prompt with more explicit rules for Grounding and applied few shot prompting, and then I got the following results which are more precise. In addition, I switched to Qwen3 model because larger models more reliably obey explicit constraints like "2 sentences only."  

In [55]:
# run on a few samples
descriptions_df = pd.read_excel("assignment_01.xlsx")
# run on 5 samples in a loop
for i in range(5):
    print(f"Evaluating sample {i}...")
    row = descriptions_df.iloc[i]
    judge_results = run_judge_task(row, row['generated_description'], client)
    print(judge_results.grounding.explanation)
    print(judge_results.grounding.verdict)

Evaluating sample 0...
All testable claims are traceable: A17 Pro chip, 120 Hz ProMotion display, USB-C fast charging, compact dimensions, titanium frame, Ceramic Shield glass, 1-year limited warranty. 'Durable titanium' is a minor embellishment based on titanium’s known properties. 'Shatter-resistant Ceramic Shield' adds a product-specific claim not in the source—this is a hallucination.
Verdict.bad
Evaluating sample 1...
All specific, testable claims (200MP camera, S-Pen support, 120Hz AMOLED, Armor Aluminum frame, Gorilla Glass Victus, sustainably sourced materials, 1-year limited warranty) are explicitly present in the source data.
Verdict.good
Evaluating sample 2...
All testable claims (Tensor G3 chip, 50MP camera, Magic Eraser, 4.7/5 rating, matte glass back, aluminum frame, 1-year limited warranty) are explicitly supported by the source data.
Verdict.good
Evaluating sample 3...
All testable claims—active noise cancelling, 30 hr battery, Bluetooth 5.2, synthetic leather earcups, 

In [ ]:
# run on full dataframe and save results
for index, row in descriptions_df.iterrows():
    judge_results = run_judge_task(row, row['generated_description'], client)
    descriptions_df.at[index, 'Fluency_llm'] = judge_results.fluency.verdict
    descriptions_df.at[index, 'Grammar_llm'] = judge_results.grammar.verdict
    descriptions_df.at[index, 'Tone_llm'] = judge_results.tone.verdict
    descriptions_df.at[index, 'Length_llm'] = judge_results.length.verdict
    descriptions_df.at[index, 'Grounding_llm'] = judge_results.grounding.verdict

descriptions_df.to_excel("assignment_01_task6.xlsx", index=False)
    

Fluency, Grammar, Tone: we are in agreement.

Length: the model does not always judge correctly. This is because the model cannot count, it only gives estimations.

Grounding: We did not always agree, but this can easily be because of my lack of knowledge about specific features implying clear derivatives. The model does seem to catch hallucinations.


Criterion-by-criterion judging:  I thought that explanations would become more specific as the full context window is dedicated to only one criterion. But after testing the most interesting critirion - Grounding - I see that all good verdicts stayed good, all bad stayed bad, and many oks were actually flagged as good.

In [ ]:
class Verdict(str, Enum):
    good = "good"
    ok = "ok"
    bad = "bad"

class CriterionResult(BaseModel):
    explanation: str = Field(..., description="1 sentence max. State the key evidence and your reasoning concisely.")
    verdict: Verdict

class JudgeResponse(BaseModel):
    # fluency: CriterionResult
    # grammar: CriterionResult
    # tone: CriterionResult
    # length: CriterionResult
    grounding: CriterionResult

def get_judge_prompt(product_row, generated_description):
    return f"""
    You are an expert E-commerce Quality Assurance Judge. 
    Evaluate the following product description based strictly on the provided rubric.

    ### SOURCE DATA:
    - Name: {product_row['product_name']}
    - Attributes: {product_row['Product_attribute_list']}
    - Material: {product_row['material']}
    - Warranty: {product_row['warranty']}

    ### GENERATED DESCRIPTION TO EVALUATE:
    "{generated_description}"

    ### RUBRIC DEFINITIONS:
    Grounding: every specific, testable claim must be traceable to the source data. 

    To evaluate, apply this test to each descriptive phrase:
    1. Is the claim specific and testable (a property, rating, or capability that could be verified or contradicted)?
     - If no (e.g. "stunning", "effortless", "ultimate"): skip, not a grounding issue.
     - If yes: proceed to step 2.
    2. Is it explicitly supported by the source data?
     - yes: grounded, fine.
     - no, but it is a well-known intrinsic property of a named material (e.g. "durable titanium"): ok, minor embellishment.
     - no, and it adds a product-specific claim not in the source (e.g. "shatter-resistant Ceramic Shield"): hallucination.

    good: all testable claims are grounded.
    ok:   one intrinsic-material embellishment; no product-specific additions.
    bad:  any product-specific claim not present in the source data.

    Example (bad): source says "Ceramic Shield glass", description says "shatter-resistant Ceramic Shield glass". 
    "Shatter-resistant" is a specific product claim absent from the source. verdict: bad.

    Example (ok): source says "titanium frame", description says "durable titanium frame". Durability is intrinsic to titanium as a material. verdict: ok.

    Provide your reasoning FIRST, then the verdict.
    """

def run_judge_task(row, description, client):
    system_msg = (
        "You are a rigorous e-commerce copy evaluator." # role
        "Evaluate product descriptions against a provided set of facts, based strictly on the provided rubric." # behavior
        "Provide detailed reasoning for each verdict." # behavior
        "Use only the product data and description supplied by the user." # gaurdrail
        "If a claim is not in the data, it is a hallucination." # gaurdrail
        "You must output a structured JSON response. For each criterion:"
        "1. Provide a clear 'explanation' first. 1 sentence only. Maximum 20 words. State the key evidence and your reasoning concisely."
        "2. Provide the 'verdict' (good, ok, or bad)." # output
    )

    prompt = get_judge_prompt(row, description)
    
    response = client.beta.chat.completions.parse( # structured response parsing
        model="Qwen/Qwen3-30B-A3B-Instruct-2507", #"google/gemma-2-9b-it-fast", #
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": prompt}
        ],
        response_format=JudgeResponse,
        max_tokens      = 600,
    )
    
    return response.choices[0].message.parsed

# run on full dataframe and save results
for index, row in descriptions_df.iterrows():
    judge_results = run_judge_task(row, row['generated_description'], client)
    descriptions_df.at[index, 'Grounding_alone_llm'] = judge_results.grounding.verdict
    descriptions_df.at[index, 'Grounding_explanation_llm'] = judge_results.grounding.explanation

descriptions_df.to_excel("assignment_01_task6.xlsx", index=False)

Analysis

- Cost: Humans need to be recruited, trained and paid. LLM is much cheaper. 
- Scale: Human evaluation does not scale. A trained evaluator can review a limited number of descriptions per hour with consistent quality, before fatigue sets in. An LLM judge processes hundreds per minute and does not get tired.
- Consistency: Human evaluators can evaluate the same description differently at 2 points in time. 2 humans can look at the same description and score it diferently. An LLM judge at temperature=0.0 is perfectly reproducible.
- Accuracy: For subjective criteria like persuasiveness, humans have edge because they bring genuine consumer intuition. They know what actually motivates a purchase. An LLM is approximating this and can be fooled. On the other hand, humans may think that a description is not grounded because of a lack of domain knowledge which an LLM can easily know.

At thousands of descriptions per day, human evaluation of every output is impractical. I would recommend using an LLM judge for every description and have human reviewers focus on descriptions that score below a threshold or that receive a bad verdict on grounding specifically.